# Shaper Pulse Comparison and Reconstruction

> Compare the Board response and pulse/energy reconstruction of generated pulses

Use measurements with and without stretcher can (shaper) on identically generated pulses to characterize the can.

## Thoughts

- Assume that the generated Pulses are "absolute" (deterministic) --> A single one recorded suffices
- record references:
    - [ ] Pulse from PGen on Oscilloscope
    - [ ] Pulse from PGen AFTER stretcher on Oscilloscope
    - [ ] Pulse from PGen on Board
    - [ ] Pulses from PGen AFTER stretcher in Board
- record through stretcher can:
    - [ ] Check PGA settings beforehand
    - [ ] Check correct impedance
    - [ ] Get help on experimental setup (Thur-25.05)
    - [ ] decide on pulse shape(s) and count
    - [ ] Determine filter settings and expected shape
    - [ ] Log generated pulses? / Dump PGen settings?
    - [ ] Vary channels from 0 to 7.
- Analysis:
    - [ ] Simulate energy reconstruction from board (expected)
    - [ ] Collect energy values from snippets
    - [ ] Average pulse shapes
    - [ ] Compare shape/energy with generated; plot them.

In [ ]:
from import_helper import *
list_imports()

from KeysightAgilent_3220A.pulse_emulation.pGen_pulse_class import (
    get_total_pulse_width,
    get_pulse_integral,
    get_pulse_function,
    get_pulse_integral_disc,
    get_total_pulse_integral,
    calculate_gap
) 

from KeysightAgilent_3220A import Agilent3220A


## Pulse-Parameters

Expected future pulse-width of stretched PMT-signals is ~250ns.
The current stretcher can has a stretching factor of ~4 (from 40 -> 150ns).

==> Use 250ns stretched width for comparability with PMT-signals (50ns un-stretched).
$$
50\text{ns base total-width}
    \approx 3\cdot 16\text{ns}
    = 48\text{ns} \Rightarrow 3\text{samples wide}
$$

### Width

Using `W = 0 := width_ns-E` and `E = 48/2 = 24 = 1.25*edge_ns`:
$$
\texttt{edge\_ns} = 24/1.25 = 19.2 = 20 \\
\texttt{width\_ns} = \texttt{E} = 24 \approx 30
$$
The `width_ns` is limited to steps of 10ns. Therefore, I use `width_ns=30`ns, which also limits `edge_ns=18.8`ns at maximum.


In [ ]:
width_ns = 30
edge_ns = 18.8


### Amplitude

Scale multiple times to try and use a larger part of the dynamic range: 300 to 4000 ADC in 5 measurements.

|Measurement|Amplitude (mV)| Amplitude (ADC)| Threshold |
|-----------|--------------|----------------|-----------|
| 1 |  11 |  300 | 625 |
| 2 |  40 | 1225 | 2419 |
| 3 |  70 | 2150 | 4276 |
| 4 | 100 | 3075 | 6132 |
| 5 | 130 | 4000 | 7988 |

(Using $x = (y-p)/m$.)

In [ ]:
def opt_threshold_for_pulse(height_mV, width, edge_ns, unit="ns"):
    if unit == "samples":
        width = width*16 - 1.25*edge_ns

    T, E, W, u = get_total_pulse_width(width, edge_ns, unit="samples")

    thr_int_mV = 1/2*height_mV*T
    # thr_int_adc= round(1/2* mV_to_adc_fit(height_mV) *T)
    thr_int_adc= round(1/2* (height_mV*30.94-28.03) *T)
    return thr_int_adc, thr_int_mV

optThresholds = []

for height_mV in [11, 40, 70, 100, 130]:
    optThresholds.append(
        opt_threshold_for_pulse(height_mV, width_ns, edge_ns)[0]
        )
optThresholds


### Burst size

1000 Cycles per measurement (Amplitude).


## Measurements


In [ ]:
group_path = "BA - ShaperCan_Characterization_1"
save_path = os.path.join(meas_base_path, group_path)


### BA230607_01

Setup: Laptop -(LAN)> PGEN -(50Ohm)> TekOsci


In [ ]:
measurement_number = "01"
attempt_number = "1"
ID = f"BA230607_{measurement_number}-{attempt_number}"

pgen = Agilent3220A()

pgen_parameters = {
    'impedance': '+5.0000000000000E+01',
    'inverted': 'INV',
    'frequency': '+1.0000000000000E+03',
    'amplitude': '+3.5000000000000E-01',
    'offset': '+0.0000000000000E+00',
    'pwidth_s': '+3.0000000000000E-08', # 30ns
    'pedge_s': '+1.8800000000000E-08',  # 18.8ns
    'burst_mode': 'TRIG',
    'ncycles': '+1.0000000000000E+00',
    'burst_state': '1'  # Burst on
    # +: set output on
    # +: set mode to pulse
    # +
}

pgen.setup()
pgen.load_parameters(pgen_parameters)
pgen.dump_parameters(os.path.join(save_path, f"{ID}.pgen.json"))


In [ ]:
pgen.do("trigger")
pgen.dump_parameters(os.path.join(save_path, f"{ID}.pgen.json"))
# save osci to tek0000

pgen.do("trigger")
# save osci to tek0001 with different view



Setup: Laptop -(LAN)> PGEN -(50Ohm)> Stretcher -> TekOsci

In [ ]:
measurement_number = "02"
attempt_number = "1"
ID = f"BA230607_{measurement_number}-{attempt_number}"

# Leave burst mode, switch to permanent
pgen.dump_parameters(os.path.join(save_path, f"{ID}.pgen.json"))
# save osci to tek0002-0007


In [ ]:
measurement_number = "03"
attempt_number = "1"
ID = f"BA230607_{measurement_number}-{attempt_number}"

pgen.dump_parameters(os.path.join(save_path, f"{ID}.pgen.json"))
# save osci to tek0002-0007
# Pulse has width of 4div -> 100ns in total
# leaves 150ns of space in width


Setup: Laptop -(LAN)> PGEN -(50Ohm)> TekOsci

In [ ]:
measurement_number = "04"
attempt_number = "1"
ID = f"BA230607_{measurement_number}-{attempt_number}"

pgen.write("function:pulse:width +6.0000000000000E-08")

pgen.do("trigger")
pgen.dump_parameters(os.path.join(save_path, f"{ID}.pgen.json"))
# record osci as tek0008

Setup: Laptop -(LAN)> PGEN -(50Ohm)> Stretcher -> TekOsci

In [ ]:
measurement_number = "05"
attempt_number = "1"
ID = f"BA230607_{measurement_number}-{attempt_number}"

pgen.write("function:pulse:width +6.0000000000000E-08")

pgen.do("trigger")
pgen.dump_parameters(os.path.join(save_path, f"{ID}.pgen.json"))
# record osci as tek0009-14


In [ ]:
measurement_number = "06"
attempt_number = "1"
ID = f"BA230607_{measurement_number}-{attempt_number}"

pgen.write("function:pulse:width +1.2000000000000E-07")

pgen.do("trigger")
pgen.dump_parameters(os.path.join(save_path, f"{ID}.pgen.json"))
# record osci as tek0015-0020


In [ ]:
measurement_number = "07"
attempt_number = "1"
ID = f"BA230607_{measurement_number}-{attempt_number}"

pgen.write("function:pulse:width +2.4000000000000E-07")

pgen.do("trigger")
pgen.dump_parameters(os.path.join(save_path, f"{ID}.pgen.json"))
# record osci as tek0021 (300ns)

In [ ]:
measurement_number = "08"
attempt_number = "1"
ID = f"BA230607_{measurement_number}-{attempt_number}"

pgen.write("function:pulse:width +2.0000000000000E-07")

pgen.do("trigger")
pgen.dump_parameters(os.path.join(save_path, f"{ID}.pgen.json"))
# record osci as tek0022-25 (250ns)

Setup: (remove stretcher)

In [ ]:
measurement_number = "09"
attempt_number = "1"
ID = f"BA230607_{measurement_number}-{attempt_number}"

pgen.write("function:pulse:width +2.0000000000000E-07")

pgen.do("trigger")
pgen.dump_parameters(os.path.join(save_path, f"{ID}.pgen.json"))
# record osci as tek0026-29, 31 (300ns)

In [ ]:
measurement_number = "10"
attempt_number = "1"
ID = f"BA230607_{measurement_number}-{attempt_number}"
pgen.dump_parameters(os.path.join(save_path, f"{ID}.pgen.json"))
# record osci as tek0032- (300ns)

In [ ]:
measurement_number = "10"
attempt_number = "2"
ID = f"BA230607_{measurement_number}-{attempt_number}"
pgen.dump_parameters(os.path.join(save_path, f"{ID}.pgen.json"))
# record osci as tek0032- (300ns)


### BA230607_11+

Setup: Laptop -(LAN)> PGEN -(50Ohm)> TekOsci


In [ ]:
measurement_number = "11"
attempt_number = "1"
ID = f"BA230607_{measurement_number}-{attempt_number}"

# pgen.load_parameters(os.path.join(save_path, f"BA230607_10-2.pgen.json"))
pgen.dump_parameters(os.path.join(save_path, f"{ID}.pgen.json"))
# save as tek0033 (without)
# tek0034 (with can)

In [ ]:
measurement_number = "12"
attempt_number = "1"
ID = f"BA230607_{measurement_number}-{attempt_number}"

pgen.write("voltage 0.10")
pgen.dump_parameters(osa.path.join(save_path, f"{ID}.pgen.json"))
# save as tek0035,36 (with)
# tek0037 (without can)

In [ ]:
measurement_number = "13"
attempt_number = "1"
ID = f"BA230607_{measurement_number}-{attempt_number}"

pgen.write("voltage 0.07")
pgen.dump_parameters(os.path.join(save_path, f"{ID}.pgen.json"))
# save as tek0038 (without)
# tek0039 (with can)

In [ ]:
measurement_number = "14"
attempt_number = "1"
ID = f"BA230607_{measurement_number}-{attempt_number}"

pgen.write("voltage 0.04")
pgen.dump_parameters(os.path.join(save_path, f"{ID}.pgen.json"))
# tek0040 (with can)
# tek0041 (without)

In [ ]:
measurement_number = "15"
attempt_number = "1"
ID = f"BA230607_{measurement_number}-{attempt_number}"

pgen.write("voltage 0.011")
pgen.dump_parameters(os.path.join(save_path, f"{ID}.pgen.json"))
# tek0042 (without)
# tek0043 (with can)